# Tool

## 使用内置Tool

In [3]:
import os
from loguru import logger
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_experimental.utilities import PythonREPL
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_community.chat_models import ChatTongyi

load_dotenv(override=True)


def debugger(x: str):
    print(x)
    return x


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你只返回纯净的 Python 代码，不要解释。代码必须是单行或多行 print。"),
        ("human", "{question}")
    ]
)

llm = ChatTongyi(
    model="qwen-plus",
    max_retries=2,
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)

# 创建Python REPL工具实例，用于执行生成的Python代码
tool = PythonREPL()

parser = StrOutputParser()
chain = prompt | llm | RunnableLambda(debugger) | parser | RunnableLambda(lambda x: tool.run(x))

result = chain.invoke({"question": "计算1到100的整数总和"})
logger.info(result)

Python REPL can execute arbitrary code. Use with caution.
2026-03-26 21:51:15.509 | INFO     | __main__:<module>:38 - 5050



content='print(sum(range(1, 101)))' additional_kwargs={} response_metadata={'model_name': 'qwen-plus', 'finish_reason': 'stop', 'request_id': '67e01941-9c05-4ce4-92b6-4e25854d52cc', 'token_usage': {'input_tokens': 45, 'output_tokens': 11, 'total_tokens': 56, 'prompt_tokens_details': {'cached_tokens': 0}}} id='lc_run--019d2a69-df48-7372-930a-998b93647b30-0' tool_calls=[] invalid_tool_calls=[]


## 自定义tool


### @tool

In [6]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool


class FieldInfo(BaseModel):
    """
    定义加法运算所需的参数信息
    """
    a: int = Field(description="第1个参数")
    b: int = Field(description="第2个参数")


@tool(name_or_callable="add", description="add two number", args_schema=FieldInfo)
def add_num(a: int, b: int) -> int:
    """
    两个整数相加
    """
    return a + b


print(add_num.name)
print(add_num.description)
print(add_num.args)
print(add_num.return_direct)

add
add two number
{'a': {'description': '第1个参数', 'title': 'A', 'type': 'integer'}, 'b': {'description': '第2个参数', 'title': 'B', 'type': 'integer'}}
False


### StructuredTool实现

In [8]:
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool


class FieldInfo(BaseModel):
    """
    定义加法运算所需的参数信息
    """
    a: int = Field(description="第1个参数")
    b: int = Field(description="第2个参数")


def add_num(a: int, b: int) -> int:
    """
    两个整数相加
    """
    return a + b


func = StructuredTool.from_function(add_num, name="add_num", args_schema=FieldInfo, return_direct=True)

print(func.name)
print(func.description)
print(func.args)
print(func.return_direct)

add_num
两个整数相加
{'a': {'description': '第1个参数', 'title': 'A', 'type': 'integer'}, 'b': {'description': '第2个参数', 'title': 'B', 'type': 'integer'}}
True


### 调用工具过程使用与分析

In [10]:
import os
from loguru import logger
from dotenv import load_dotenv
from datetime import date
from langchain_core.tools import tool
from langchain_community.chat_models import ChatTongyi

load_dotenv(override=True)


@tool
def get_today() -> str:
    """
    获取当前系统日期

    Returns:
        str: 今天的日期字符串，格式为 yyyy-MM-dd
    """
    logger.info("执行工具：get_today")
    return date.today().isoformat()


llm = ChatTongyi(
    model="qwen-plus",
    max_retries=2,
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)
# 将工具绑定到语言模型
llm_with_tools = llm.bind_tools([get_today])
# 用户提问
question_list = ["你是谁？", "今天是几号？"]
for question in question_list:
    logger.info(f"用户问题：{question}")
    # 调用语言模型处理用户问题
    ai_msg = llm_with_tools.invoke(question)
    logger.info(f"LLM回复：{ai_msg}")
    # 检查是否有工具调用
    if ai_msg.tool_calls:
        logger.info(ai_msg.tool_calls)
        # 获取第一个工具调用信息
        tool_call = ai_msg.tool_calls[0]
        # 执行对应的工具函数并获取结果
        # locals() = 当前所有变量 / 函数的字典, locals()["函数名"] = 把字符串变成真正可调用的函数
        tool_result = locals()[tool_call["name"]].invoke(tool_call["args"])
        logger.info(f"调用工具结果：{tool_result}")
    else:
        # 直接输出语言模型的回答
        logger.info(f"LLM 直接作答：{ai_msg.content}")

2026-03-26 22:16:33.272 | INFO     | __main__:<module>:33 - 用户问题：你是谁？
2026-03-26 22:16:35.035 | INFO     | __main__:<module>:36 - LLM回复：content='我是通义千问，阿里巴巴集团旗下的超大规模语言模型。我能够回答问题、创作文字，比如写故事、写公文、写邮件、写剧本、逻辑推理、编程等等，还能表达观点，玩游戏等。如果你有任何问题或需要帮助，欢迎随时告诉我！' additional_kwargs={} response_metadata={'model_name': 'qwen-plus', 'finish_reason': 'stop', 'request_id': 'f619b05c-b4d5-4bfc-ba07-d1c903c4b064', 'token_usage': {'input_tokens': 150, 'output_tokens': 60, 'total_tokens': 210, 'prompt_tokens_details': {'cached_tokens': 0}}} id='lc_run--019d2a81-0af9-7082-964b-d02f5ddf94d6-0' tool_calls=[] invalid_tool_calls=[]
2026-03-26 22:16:35.036 | INFO     | __main__:<module>:48 - LLM 直接作答：我是通义千问，阿里巴巴集团旗下的超大规模语言模型。我能够回答问题、创作文字，比如写故事、写公文、写邮件、写剧本、逻辑推理、编程等等，还能表达观点，玩游戏等。如果你有任何问题或需要帮助，欢迎随时告诉我！
2026-03-26 22:16:35.036 | INFO     | __main__:<module>:33 - 用户问题：今天是几号？
2026-03-26 22:16:35.858 | INFO     | __main__:<module>:36 - LLM回复：content='' additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_f3d10fb3d34c47